# Introduction to Machine Learning

This lab follows the lecture on the basics of machine learning and has the same four parts:

1. **Data** – samples, features, targets, labeled and unlabeled data; the basics of working with a table in pandas.
2. **Principles** – the model as a function learned from data, `fit` / `predict`, training, validation and test data, underfitting and overfitting.
3. **Tasks** – regression, classification and clustering on real datasets.
4. **Evaluation** – baselines, metrics for each task and why accuracy alone can be misleading.

We use the algorithms from **scikit-learn** as ready-made black boxes; how they work inside is covered in later labs.
The same goes for **data preprocessing** (lab 08) and **systematic model selection** – tuning hyperparameters by a metric, cross-validation, choosing a decision threshold (lab 07).
That is why today we only use built-in datasets that are already clean.

The code is commented line by line. Cells marked **✏️ Exercise** are for you to complete during the lab;
the second half of the lab is [independent work](Independent_Work_EN.ipynb).

In [ ]:
import numpy as np                     # arrays and maths
import pandas as pd                    # DataFrame tables and Series columns
import matplotlib.pyplot as plt        # plots and images
from IPython.display import display    # nicely formatted tables in the middle of a cell

from sklearn import datasets, metrics                                     # built-in data and metrics
from sklearn.model_selection import train_test_split                      # splitting data
from sklearn.dummy import DummyRegressor, DummyClassifier                 # trivial reference models
from sklearn.linear_model import LinearRegression                         # linear regression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier    # decision trees
from sklearn.ensemble import RandomForestClassifier                       # random forest
from sklearn.cluster import KMeans                                        # clustering

RSTATE = 42  # fixed random seed -> reproducible results

# Part 1 – Data

Machine learning learns from **data**. A typical (tabular) dataset is a table where

* each **row** is one **sample** (an observation, an example) – e.g. one patient,
* each **column** is one **feature** (an attribute, an input variable) – e.g. the size of a tumour,
* optionally one special column is the **target** (label, output) – the value we want to predict.

We write the inputs as a matrix $X$ of shape `(n_samples, n_features)` and the targets as a vector $y$ of length `n_samples`.

## 1.1 A tabular dataset – Breast Cancer Wisconsin

Each sample describes cell nuclei measured on a microscope image of a breast tissue sample (569 samples, 30 numerical features).
The target says whether the tumour is *malignant* or *benign*.

With `as_frame=True` we get the data as **pandas** objects:

* `cancer.data` – the features only (**DataFrame** = a table with column names and a row index),
* `cancer.target` – the target only (**Series** = a single column with an index).

In this dataset `0 = malignant` and `1 = benign`. It is common to encode the class we are looking for (the disease) as `1` – the **positive class** – so we flip the encoding.

In [ ]:
cancer = datasets.load_breast_cancer(as_frame=True)    # load the data as pandas objects
X_cancer = cancer.data                                  # DataFrame with 30 features
y_cancer = (cancer.target == 0).astype(int)             # Series: 1 = malignant (positive class), 0 = benign

df = X_cancer.assign(target=y_cancer)                   # new table: features + target column (only for browsing)
print("X shape:", X_cancer.shape, "  y shape:", y_cancer.shape, "  df shape:", df.shape)
display(df.head(3))                                     # first three rows

## 1.2 pandas in a nutshell

A few commands are enough for browsing data:

| command | returns |
|---|---|
| `df.shape`, `df.head()`, `df.dtypes` | dimensions, first rows, data type of each column |
| `df["a"]` / `df[["a", "b"]]` | one column as a Series / selected columns as a DataFrame |
| `df.loc[mask, ["a", "b"]]` | rows where the mask is `True`, selected columns |
| `(cond1) & (cond2)`, `|`, `~` | "and", "or", "not" for masks (put each condition in parentheses) |
| `y.value_counts(normalize=True)` | counts (shares) of each value |
| `df.describe()` | count, mean, standard deviation, minimum, quartiles, maximum |
| `df.drop(columns="target")` | the table without the given column |

Note: the target column **must not** be among the model inputs – the model would just copy the right answer.

In [ ]:
radius = df["mean radius"]                     # one column -> Series, shape (n,)
radius_tab = df[["mean radius"]]               # a list of columns -> DataFrame, shape (n, 1)
print(type(radius).__name__, radius.shape, "|", type(radius_tab).__name__, radius_tab.shape)

mask = (df["target"] == 1) & (df["mean radius"] > 20)           # malignant and at the same time large tumours
print("Selected rows:", mask.sum())                              # True counts as 1 when summing
display(df.loc[mask, ["mean radius", "mean area", "target"]].head())    # rows by mask, selected columns

In [ ]:
counts = pd.DataFrame({                                          # a table from two Series with the same index
    "count": y_cancer.value_counts(),                            # number of samples in each class
    "share": y_cancer.value_counts(normalize=True).round(3),     # the same as a share
}).rename(index={0: "benign (0)", 1: "malignant (1)"})           # readable names instead of codes
display(counts)

stats = X_cancer.describe().T                                    # .T transposes the table: features in rows
display(stats.loc[["mean radius", "mean area", "mean smoothness"], ["mean", "std", "min", "max"]])

**✏️ Exercise 1.1** – answer from the outputs above:

* How many samples and how many features does the dataset have? Are all features numerical?
* Is the dataset *balanced* (are both classes represented equally)?
* Compare the ranges of `mean area` and `mean smoothness`. (Some algorithms are sensitive to different ranges – we will deal with that in lab 08; the models used today are not.)
* Change the mask to select **benign** tumours with `mean radius` greater than 15. How many are there?

## 1.3 Not only tables – images are data too

A greyscale image of $8 \times 8$ pixels is just $64$ numbers. If we *flatten* it, each pixel becomes one feature and the image becomes one row of the matrix $X$.

In [ ]:
digits = datasets.load_digits()                                   # 1797 handwritten digits 0–9
print("X shape:", digits.data.shape, "  (1797 images, 64 pixels each)")

fig, axes = plt.subplots(1, 10, figsize=(12, 2))                  # ten images side by side
for ax, img, label in zip(axes, digits.images, digits.target):    # image and its correct class
    ax.imshow(img, cmap="gray_r")                                 # 0 = white, 16 = black
    ax.set_title(f"y = {label}")
    ax.axis("off")                                                # no axes
plt.show()

print(digits.images[0].astype(int))   # the first image as an 8 × 8 matrix ...
print(digits.data[0].astype(int))     # ... and as one row of the matrix X (64 features)

## 1.4 Labeled vs. unlabeled data

* **Supervised learning** – every sample has a known target $y$. The model learns the mapping $X \rightarrow y$ (regression, classification).
* **Unsupervised learning** – we only have $X$. The model looks for structure in the data itself (clustering, dimensionality reduction).

Both plots show the same two features. On the left we know the labels, on the right we do not – that is all an unsupervised algorithm can see.

In [ ]:
f1, f2 = "mean radius", "mean texture"                            # two features for a 2D plot

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for cls, name, color in [(0, "benign", "tab:blue"), (1, "malignant", "tab:red")]:
    m = y_cancer == cls                                           # mask of one class
    axes[0].scatter(X_cancer.loc[m, f1], X_cancer.loc[m, f2], s=12, alpha=0.6, c=color, label=name)
axes[0].set_title("Labeled data (supervised learning)")
axes[0].legend()

axes[1].scatter(X_cancer[f1], X_cancer[f2], s=12, alpha=0.6, c="gray")   # without the target
axes[1].set_title("Unlabeled data (unsupervised learning)")

for ax in axes:
    ax.set_xlabel(f1)
axes[0].set_ylabel(f2)
plt.tight_layout()
plt.show()

# Part 2 – Principles

## 2.1 A model is a function with learnable parameters

A model is a function $\hat{y} = f(x; \theta)$. **Learning** (training) means finding parameters $\theta$ so that the predictions $\hat{y}$ are close to the true targets $y$
on the training data – i.e. minimising a **loss function**.

* **Parameters** are learned from data (e.g. the weights of a linear model).
* **Hyperparameters** are chosen by us before training (e.g. the maximal depth of a decision tree).

In scikit-learn every model has the same interface:

```python
model = SomeModel(hyperparameters)   # 1. choose a model
model.fit(X_train, y_train)          # 2. learn the parameters from data
y_pred = model.predict(X_new)        # 3. use the model on new data
```

A minimal example: we generate data from $y = 3x + 2 + \text{noise}$ and let linear regression find the parameters.

In [ ]:
rng = np.random.default_rng(RSTATE)               # reproducible random number generator
x = rng.uniform(0, 10, size=50)                   # 50 inputs from the interval 0–10
y = 3 * x + 2 + rng.normal(0, 2, size=50)         # true relationship + noise

model = LinearRegression()                        # 1. choose a model
model.fit(x.reshape(-1, 1), y)                    # 2. learn; sklearn expects X as a 2D matrix (samples, features)
print(f"Learned parameters: slope = {model.coef_[0]:.2f}, intercept = {model.intercept_:.2f}")
print("Prediction for x = 4:", model.predict([[4]]))   # 3. prediction for a new sample

xs = np.linspace(0, 10, 100).reshape(-1, 1)       # dense grid for drawing the line
plt.scatter(x, y, s=15, label="data")
plt.plot(xs, model.predict(xs), "r", label="learned model")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.show()

## 2.2 Generalization and the train/test split

We do not want a model that is good on the data it has already seen – we want a model that works on **new, unseen data**. This ability is called **generalization**.

To estimate it, we hide part of the data before training:

* **training set** – the model learns from it,
* **test set** – the model is only evaluated on it; it must not see it during training.

We split `X` and `y` with **one** call of `train_test_split`, so the rows of inputs and answers stay paired. `stratify=y` keeps the class ratio the same in both parts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(              # 80 % training, 20 % test data
    X_cancer, y_cancer, test_size=0.2, stratify=y_cancer, random_state=RSTATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Share of malignant – train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")  # mean of 0/1 = share of ones

## 2.3 Underfitting and overfitting

* **Underfitting** – the model is too simple, it cannot capture the pattern. It is bad on training *and* test data.
* **Overfitting** – the model is too complex, it memorises the training data including noise. It is excellent on training data but bad on test data.

Example: 20 noisy training points (and 200 test points from the same range) generated from a sine wave, fitted by polynomials of increasing degree (= increasing model complexity).

In [ ]:
rng = np.random.default_rng(1)                                     # separate generator for this example
def true_fn(x):                                                    # true relationship without noise
    return np.sin(2 * np.pi * x)

x_tr = np.linspace(0, 1, 20);  y_tr = true_fn(x_tr) + rng.normal(0, 0.25, 20)     # small training set
x_te = rng.uniform(0, 1, 200); y_te = true_fn(x_te) + rng.normal(0, 0.25, 200)    # independent test set
xs = np.linspace(0, 1, 500)                                        # grid for smooth curves

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree, title in zip(axes, [1, 4, 15], ["underfitting", "good fit", "overfitting"]):
    poly = np.polynomial.Polynomial.fit(x_tr, y_tr, degree)        # fit a polynomial of the given degree
    mse_tr = np.mean((poly(x_tr) - y_tr) ** 2)                     # mean squared error on training data
    mse_te = np.mean((poly(x_te) - y_te) ** 2)                     # ... and on test data
    ax.scatter(x_tr, y_tr, s=20, label="training data")
    ax.plot(xs, true_fn(xs), "g--", label="true function")
    ax.plot(xs, poly(xs), "r", label=f"polynomial, degree {degree}")
    ax.set_ylim(-2, 2)                                             # curves outside the range are clipped
    ax.set_title(f"{title}\ntrain MSE = {mse_tr:.3f}, test MSE = {mse_te:.3f}")
    ax.legend(loc="lower left", fontsize=8)
plt.show()

## 2.4 Choosing model complexity – why we need a validation set

How do we pick a good complexity (a hyperparameter)? **Not by looking at the test set** – then it would no longer be "unseen" and our estimate would be too optimistic.
So we split the training data once more:

| part | used for |
|---|---|
| **train** | learning the model parameters |
| **validation** | comparing settings (hyperparameters) and models |
| **test** | the final estimate of performance on unseen data |

Example: decision trees with different maximal depth on the digits dataset. We only **read the plot by eye** – systematic model selection by a metric and cross-validation are covered in lab 07.

In [ ]:
Xg_train, Xg_test, yg_train, yg_test = train_test_split(           # 1. put the test set aside
    digits.data, digits.target, test_size=0.2, stratify=digits.target, random_state=RSTATE
)
X_tr, X_val, y_tr, y_val = train_test_split(                       # 2. split the rest into train and validation
    Xg_train, yg_train, test_size=0.25, stratify=yg_train, random_state=RSTATE
)

depths = range(1, 21)                                              # maximal depths to try
acc_tr, acc_val = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=RSTATE).fit(X_tr, y_tr)   # learn on training data only
    acc_tr.append(tree.score(X_tr, y_tr))                          # .score() = accuracy
    acc_val.append(tree.score(X_val, y_val))                       # accuracy on validation data

plt.plot(depths, acc_tr, "o-", label="training data")
plt.plot(depths, acc_val, "o-", label="validation data")
plt.xticks(list(depths))
plt.xlabel("max_depth (model complexity)"); plt.ylabel("accuracy")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**✏️ Exercise 2.1**

* Where is the model underfitting and where is it overfitting? From which depth does the validation accuracy stop improving?
* Why does the training accuracy reach 1.0 for deep trees? Does the validation accuracy follow it?
* Change `random_state` in the train/validation split. Does the shape of the curves change? What does it tell you about a single validation split?

# Part 3 – Tasks

| task | type | target | example |
|---|---|---|---|
| **Regression** | supervised | continuous number | price of a house, disease progression |
| **Classification** | supervised | discrete class | malignant / benign, digit 0–9 |
| **Clustering** | unsupervised | none | groups of similar customers |

Other tasks you may meet: dimensionality reduction, anomaly detection, recommendation, generative modelling, reinforcement learning.

## 3.1 Regression – Diabetes dataset

10 baseline measurements (age, sex, BMI, blood pressure, six blood serum values) of 442 patients. The target is a quantitative measure of disease progression one year later.
The features are already standardised by the authors of the dataset.

In [ ]:
diabetes = datasets.load_diabetes(as_frame=True)                  # regression data as pandas
X_diab, y_diab = diabetes.data, diabetes.target                   # inputs and a continuous target
display(X_diab.assign(target=y_diab).head(3))                     # look at a few rows

Xd_train, Xd_test, yd_train, yd_test = train_test_split(          # no stratification for a continuous target
    X_diab, y_diab, test_size=0.2, random_state=RSTATE
)
lin_reg = LinearRegression().fit(Xd_train, yd_train)                                  # linear model
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=RSTATE).fit(Xd_train, yd_train)  # tree of depth 3

pd.DataFrame({                                                    # true values and predictions side by side
    "true value": yd_test.values[:5],
    "linear regression": lin_reg.predict(Xd_test[:5]).round(1),
    "decision tree": tree_reg.predict(Xd_test[:5]).round(1),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, model, name in zip(axes, [lin_reg, tree_reg], ["Linear regression", "Decision tree (depth 3)"]):
    ax.scatter(yd_test, model.predict(Xd_test), s=15, alpha=0.7)  # true value vs. prediction
    lims = [y_diab.min(), y_diab.max()]
    ax.plot(lims, lims, "r--", label="perfect prediction")        # points on the diagonal = no error
    ax.set_title(name); ax.set_xlabel("true value"); ax.legend()
axes[0].set_ylabel("predicted value")
plt.show()

The closer the points are to the red diagonal, the better the predictions. *How much* better one model is than the other we will measure in Part 4.

A decision tree of depth 3 has at most $2^3 = 8$ leaves, so it can predict at most 8 different values – that is why you can see the horizontal "stripes".

## 3.2 Classification

### Binary classification – Breast Cancer

Classifiers return not only the class (`predict`) but often also an **estimate of the probability** of each class (`predict_proba`).
The columns of `predict_proba` are in the same order as `model.classes_`.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=RSTATE).fit(X_train, y_train)  # forest of 200 trees
print("Class order:", forest.classes_)                            # [0 1] -> column 1 belongs to class 1 (malignant)

pd.DataFrame({
    "true class": y_test.values[:8],                              # correct answers
    "predicted class": forest.predict(X_test[:8]),                # predicted classes
    "P(malignant)": forest.predict_proba(X_test[:8])[:, 1].round(3),  # estimated probability of class 1
})

### Multiclass classification – handwritten digits

The same interface works for 10 classes. We reuse the digits split from Part 2 and look at the images the model got wrong.

In [ ]:
digit_clf = RandomForestClassifier(n_estimators=200, random_state=RSTATE).fit(Xg_train, yg_train)
yg_pred = digit_clf.predict(Xg_test)                              # predictions for the test images

wrong = np.where(yg_pred != yg_test)[0]                           # positions of misclassified images
print(f"Wrongly classified: {len(wrong)} of {len(yg_test)} test images")

fig, axes = plt.subplots(1, min(10, len(wrong)), figsize=(12, 2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(Xg_test[i].reshape(8, 8), cmap="gray_r")            # a row of 64 pixels back to an 8 × 8 image
    ax.set_title(f"true {yg_test[i]}\npred {yg_pred[i]}", fontsize=9)
    ax.axis("off")
plt.show()

## 3.3 Clustering – Iris without labels

The Iris dataset contains measurements of 150 iris flowers of three species. We **hide the species** and ask K-means to find 3 groups.
(K-means itself is explained in detail in lab 06.)

`pd.crosstab` counts how many flowers of each species ended up in each cluster.

In [ ]:
iris = datasets.load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target                           # the species stay aside

kmeans = KMeans(n_clusters=3, n_init=10, random_state=RSTATE)     # 3 clusters, 10 random starts
clusters = kmeans.fit_predict(X_iris)                             # note: no y here!

fx, fy = "petal length (cm)", "petal width (cm)"                  # two of the four features for the plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axes[0].scatter(X_iris[fx], X_iris[fy], c=clusters, cmap="viridis", s=20)
axes[0].set_title("Clusters found by K-means")
axes[1].scatter(X_iris[fx], X_iris[fy], c=y_iris, cmap="viridis", s=20)
axes[1].set_title("True species (unknown to K-means)")
for ax in axes:
    ax.set_xlabel(fx)
axes[0].set_ylabel(fy)
plt.show()

species = pd.Series(iris.target_names[y_iris], name="species")    # species names only to describe the result
pd.crosstab(species, pd.Series(clusters, name="cluster"))         # species × cluster

The cluster numbers are arbitrary – K-means does not know the names of the species, it only groups similar flowers.
The table shows that *setosa* is separated perfectly while *versicolor* and *virginica* partly overlap.

**✏️ Exercise 3.1**

* For each problem decide whether it is regression, classification or clustering:
  predicting tomorrow's temperature, spam filtering, grouping customers by shopping behaviour, recognising a road sign, estimating the remaining life of a bearing.
* Run K-means on Iris with `n_clusters=2` and `n_clusters=5`. Is there a "correct" number of clusters when we have no labels?

# Part 4 – Evaluation

A number like "accuracy 0.95" means nothing on its own. We always ask: **compared to what?** and **is it the right metric for the problem?**

## 4.1 Baseline – the simplest possible model

A **baseline** is a trivial model that ignores the features:

* regression – always predict the mean of the training targets (`DummyRegressor`),
* classification – always predict the most frequent class (`DummyClassifier`).

A useful model must be clearly better than the baseline.

## 4.2 Regression metrics

| metric | formula | meaning |
|---|---|---|
| MAE | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | average error, in units of the target |
| RMSE | $\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$ | like MAE, but large errors are penalised more |
| $R^2$ | $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | 1 = perfect, 0 = as good as predicting the mean, < 0 = worse than the mean |

In [ ]:
dummy_reg = DummyRegressor(strategy="mean").fit(Xd_train, yd_train)   # always predicts the mean of training targets

rows = []                                                         # rows of the results table
for name, model in [("baseline (mean)", dummy_reg), ("linear regression", lin_reg), ("decision tree", tree_reg)]:
    for split, X_, y_ in [("train", Xd_train, yd_train), ("test", Xd_test, yd_test)]:
        p = model.predict(X_)                                     # predictions for the given part of the data
        rows.append({
            "model": name, "data": split,
            "MAE": metrics.mean_absolute_error(y_, p),
            "RMSE": metrics.root_mean_squared_error(y_, p),
            "R2": metrics.r2_score(y_, p),
        })
pd.DataFrame(rows).set_index(["model", "data"]).round(2)          # table with a two-level index

## 4.3 Classification metrics

For a binary classifier every prediction falls into one of four cells of the **confusion matrix**:

| | predicted negative | predicted positive |
|---|---|---|
| **actual negative** | TN – true negative | FP – false positive (false alarm) |
| **actual positive** | FN – false negative (missed case) | TP – true positive |

| metric | formula | question it answers |
|---|---|---|
| accuracy | $\frac{TP + TN}{TP + TN + FP + FN}$ | how often is the model right? |
| precision | $\frac{TP}{TP + FP}$ | when the model says "positive", how often is it right? |
| recall (sensitivity) | $\frac{TP}{TP + FN}$ | how many of the actual positives did the model find? |
| F1 | $2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$ | harmonic mean of precision and recall |

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)   # always the most frequent class
predictions = {"baseline": dummy_clf.predict(X_test), "random forest": forest.predict(X_test)}

rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, p) in zip(axes, predictions.items()):
    metrics.ConfusionMatrixDisplay.from_predictions(              # confusion matrix from existing predictions
        y_test, p, labels=[0, 1], display_labels=["benign", "malignant"], ax=ax, colorbar=False
    )
    ax.set(title=name, xlabel="predicted class", ylabel="true class")
    rows.append({                                                 # metrics for the positive class 1 = malignant
        "model": name,
        "accuracy": metrics.accuracy_score(y_test, p),
        "precision": metrics.precision_score(y_test, p, zero_division=0),
        "recall": metrics.recall_score(y_test, p),
        "F1": metrics.f1_score(y_test, p, zero_division=0),
    })
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("model").round(3))

The baseline has an accuracy of over 60 % – without looking at a single feature. And its recall is 0: it never finds a malignant tumour.

**Which error is worse here?** A false positive means an unnecessary extra examination. A false negative means a missed cancer.
For this problem **recall** of the malignant class is the critical metric. The choice of metric is a decision about the *problem*, not about the model.

## 4.4 Why accuracy can lie – imbalanced data

Imagine a fault-detection dataset where only 3 % of the samples are faults.

In [ ]:
X_imb, y_imb = datasets.make_classification(                      # a synthetic binary task
    n_samples=5000, n_features=10, n_informative=4, weights=[0.97, 0.03], random_state=RSTATE
)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=RSTATE)
print(f"Share of positive class (faults): {y_imb.mean():.3f}")

rows = []
for name, model in [("baseline", DummyClassifier(strategy="most_frequent")),
                    ("random forest", RandomForestClassifier(random_state=RSTATE))]:
    p = model.fit(Xi_train, yi_train).predict(Xi_test)            # learn and predict in one line
    rows.append({"model": name,
                 "accuracy": metrics.accuracy_score(yi_test, p),
                 "recall": metrics.recall_score(yi_test, p),
                 "F1": metrics.f1_score(yi_test, p, zero_division=0)})
display(pd.DataFrame(rows).set_index("model").round(3))

A model that **never** detects a fault has an accuracy of about 97 %. The random forest is only slightly better in accuracy, but it actually finds some of the faults.
On imbalanced data always look at the confusion matrix, precision and recall (or F1), not just accuracy.

**✏️ Exercise 4.1** – For cancer screening, recall matters most. For which problem would precision matter more? (Hint: a spam filter and an important e-mail in the spam folder.)

## 4.5 Evaluating clustering

Without labels there is no "correct answer" to compare against. We can only measure how compact and well separated the clusters are:

* **inertia** – sum of squared distances of the points to their cluster centre (lower = more compact, but always decreases with more clusters),
* **silhouette score** – from −1 to 1, higher = points are close to their own cluster and far from the others.

If the true labels happen to be known (as for Iris), we can also compare with them, e.g. using the **Adjusted Rand Index** (1 = identical grouping, ≈ 0 = random).

In [ ]:
rows = []
for k in range(2, 11):                                            # different numbers of clusters
    km = KMeans(n_clusters=k, n_init=10, random_state=RSTATE).fit(X_iris)   # learning without species
    rows.append({"k": k,
                 "inertia": km.inertia_,
                 "silhouette": metrics.silhouette_score(X_iris, km.labels_),
                 "ARI": metrics.adjusted_rand_score(y_iris, km.labels_)})   # the only metric using the true species
cluster_results = pd.DataFrame(rows).set_index("k")

cluster_results.plot(subplots=True, layout=(1, 3), figsize=(15, 4), marker="o", legend=False,
                     title=["Inertia", "Silhouette score", "Adjusted Rand Index (uses true species)"])
plt.tight_layout()
plt.show()

Interesting: the silhouette score is highest for **2** clusters, because *versicolor* and *virginica* overlap, while the true number of species is 3.
Unsupervised metrics measure the geometry of the data, not "the truth".

# Summary – a checklist for every ML experiment

1. **Data** – What is a sample? What are the features and the target? Is it labeled? Is it balanced?
2. **Task** – Regression, classification or clustering?
3. **Split** – Put the test data aside before you start tuning anything. Compare settings on validation data.
4. **Baseline** – How good is a trivial model?
5. **Metric** – Which errors matter for *this* problem? Choose the metric accordingly.
6. **Generalization** – Compare training and test performance: underfitting or overfitting?

**Later in the semester:** data preprocessing (lab 08); systematic model selection by a validation metric, cross-validation and choosing a decision threshold (lab 07).

Continue with the [independent work](Independent_Work_EN.ipynb).